In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, quarter, sum, count, avg

spark = (
    SparkSession.builder
    .appName("Sales Analytics ETL")
    .enableHiveSupport()
    .getOrCreate()
)

base_path = "hdfs://namenode:9000/data/raw/sales"

sales_df = (
    spark.read.option("header", True)
    .csv(f"{base_path}/sales_transactions.csv")
    .withColumnRenamed("SalesID", "sales_id")
    .withColumnRenamed("ProductID", "product_id")
    .withColumnRenamed("RegionID", "region_id")
    .withColumnRenamed("QuantitySold", "quantity_sold")
    .withColumnRenamed("SalesAmount", "sales_amount")
    .withColumnRenamed("SalesDate", "sales_date")
    .withColumn("sales_id", col("sales_id").cast("int"))
    .withColumn("product_id", col("product_id").cast("int"))
    .withColumn("region_id", col("region_id").cast("int"))
    .withColumn("quantity_sold", col("quantity_sold").cast("int"))
    .withColumn("sales_amount", col("sales_amount").cast("double"))
    .withColumn("sales_date", to_date(col("sales_date"), "yyyy-MM-dd"))
)

products_df = (
    spark.read.option("header", True)
    .csv(f"{base_path}/products.csv")
    .withColumnRenamed("ProductID", "product_id")
    .withColumnRenamed("ProductName", "product_name")
    .withColumnRenamed("CategoryID", "category_id")
    .withColumn("product_id", col("product_id").cast("int"))
    .withColumn("category_id", col("category_id").cast("int"))
)

categories_df = (
    spark.read.option("header", True)
    .csv(f"{base_path}/categories.csv")
    .withColumnRenamed("CategoryID", "category_id")
    .withColumnRenamed("CategoryName", "category_name")
    .withColumn("category_id", col("category_id").cast("int"))
)

regions_df = (
    spark.read.option("header", True)
    .csv(f"{base_path}/regions.csv")
    .withColumnRenamed("RegionID", "region_id")
    .withColumnRenamed("RegionName", "region_name")
    .withColumn("region_id", col("region_id").cast("int"))
)

final_df = (
    sales_df
    .join(products_df, "product_id", "left")
    .join(categories_df, "category_id", "left")
    .join(regions_df, "region_id", "left")
    .withColumn("sales_year", year(col("sales_date")))
    .withColumn("sales_month", month(col("sales_date")))
    .withColumn("sales_quarter", quarter(col("sales_date")))
)

final_df.printSchema()
final_df.show(10, truncate=False)

spark.sql("CREATE DATABASE IF NOT EXISTS sales_dw")
spark.sql("USE sales_dw")

final_df.write.mode("overwrite").format("parquet").saveAsTable("fact_sales_analytics")

summary_by_region = (
    final_df.groupBy("region_name")
    .agg(
        count("sales_id").alias("total_transactions"),
        sum("quantity_sold").alias("total_quantity"),
        sum("sales_amount").alias("total_sales"),
        avg("sales_amount").alias("avg_sales")
    )
    .orderBy(col("total_sales").desc())
)

summary_by_region.write.mode("overwrite").format("parquet").saveAsTable("summary_sales_by_region")

summary_by_product = (
    final_df.groupBy("product_name", "category_name")
    .agg(
        count("sales_id").alias("total_transactions"),
        sum("quantity_sold").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    )
    .orderBy(col("total_sales").desc())
)

summary_by_product.write.mode("overwrite").format("parquet").saveAsTable("summary_sales_by_product")

summary_by_month = (
    final_df.groupBy("sales_year", "sales_month")
    .agg(
        count("sales_id").alias("total_transactions"),
        sum("quantity_sold").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    )
    .orderBy("sales_year", "sales_month")
)

summary_by_month.write.mode("overwrite").format("parquet").saveAsTable("summary_sales_by_month")

print("ETL completed successfully.")

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS sales_dw")
spark.sql("USE sales_dw")

final_df.write.mode("overwrite").saveAsTable("fact_sales_analytics")

In [ ]:
spark.sql("USE sales_dw")

final_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("sales_dw.fact_sales_analytics")

spark.sql("SHOW TABLES IN sales_dw").show()

In [ ]:
spark.sql("USE sales_dw")

final_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "hdfs://namenode:9000/user/hive/warehouse/sales_dw.db/fact_sales_analytics") \
    .saveAsTable("sales_dw.fact_sales_analytics")

In [ ]:
# ===============================
# DATA VISUALIZATION SECTION
# ===============================

import matplotlib.pyplot as plt
import os

output_dir = "/home/jovyan/work/outputs/charts"
os.makedirs(output_dir, exist_ok=True)

# Read Hive summary tables
region_pdf = spark.sql("""
SELECT region_name, total_transactions, total_quantity, total_sales, avg_sales
FROM sales_dw.summary_sales_by_region
ORDER BY total_sales DESC
""").toPandas()

product_pdf = spark.sql("""
SELECT product_name, category_name, total_quantity, total_sales
FROM sales_dw.summary_sales_by_product
ORDER BY total_sales DESC
LIMIT 10
""").toPandas()

month_pdf = spark.sql("""
SELECT sales_year, sales_month, total_sales, total_quantity, total_transactions
FROM sales_dw.summary_sales_by_month
ORDER BY sales_year, sales_month
""").toPandas()

# Create month label
month_pdf["month_label"] = (
    month_pdf["sales_year"].astype(str) + "-" +
    month_pdf["sales_month"].astype(str).str.zfill(2)
)

# ===============================
# 1. Sales by Region
# ===============================
plt.figure(figsize=(10, 6))
plt.bar(region_pdf["region_name"], region_pdf["total_sales"])
plt.title("Total Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{output_dir}/01_total_sales_by_region.png", dpi=300)
plt.show()

# ===============================
# 2. Top 10 Products by Sales
# ===============================
plt.figure(figsize=(12, 6))
plt.bar(product_pdf["product_name"], product_pdf["total_sales"])
plt.title("Top 10 Products by Total Sales")
plt.xlabel("Product")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{output_dir}/02_top_10_products_by_sales.png", dpi=300)
plt.show()

# ===============================
# 3. Monthly Sales Trend
# ===============================
plt.figure(figsize=(12, 6))
plt.plot(month_pdf["month_label"], month_pdf["total_sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{output_dir}/03_monthly_sales_trend.png", dpi=300)
plt.show()

# ===============================
# 4. Quantity Sold by Region
# ===============================
plt.figure(figsize=(10, 6))
plt.bar(region_pdf["region_name"], region_pdf["total_quantity"])
plt.title("Total Quantity Sold by Region")
plt.xlabel("Region")
plt.ylabel("Quantity Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{output_dir}/04_quantity_sold_by_region.png", dpi=300)
plt.show()

# ===============================
# 5. Transactions by Month
# ===============================
plt.figure(figsize=(12, 6))
plt.plot(month_pdf["month_label"], month_pdf["total_transactions"], marker="o")
plt.title("Monthly Transaction Trend")
plt.xlabel("Month")
plt.ylabel("Total Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{output_dir}/05_monthly_transaction_trend.png", dpi=300)
plt.show()

print("Data visualization completed successfully.")
print(f"Charts saved to: {output_dir}")